# v1 — Basic Bigram Language Model

This is the first step in building a GPT-style transformer from scratch. We start with the simplest possible language model: a **bigram model**, which predicts the next character using *only* the current character — no context, no attention, just a lookup table.

We'll train it character-by-character on the Harry Potter text.

## 1. Imports

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 2. Hyperparameters

- `batch_size`: how many independent sequences we process in parallel
- `block_size`: the maximum context length used for predictions
- `device`: train on GPU if available, otherwise CPU

In [3]:
batch_size = 32      # how many independent sequences will we process in parallel?
block_size = 8       # what is the maximum context length for predictions?
max_iters = 10000    # maximum number of iterations for training
eval_interval = 500  # interval for evaluating the model
learning_rate = 1e-3 # learning rate for training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200     # number of iterations for evaluation
seed = 42            # seed for reproducibility

torch.manual_seed(seed)

## 3. Load the dataset

We'll train on the Harry Potter text.

In [4]:
with open('./data/harry_potter.txt', encoding='utf-8') as f:
    text = f.read()

print(f"length of dataset in characters: {len(text)}")
print(text[:500])

length of dataset in characters: 5991293
THE BOY WHO LIVED Mr and Mrs Dursley of number four Privet Drive were proud to say that they were perfectly normal thank you very much .They were the last people youd expect to be involved in anything strange or mysterious because they just didnt hold with such nonsense .Mr Dursley was the director of a firm called Grunnings which made drills .He was a big beefy man with hardly any neck although he did have a very large mustache .Mrs Dursley was thin and blonde and had nearly twice the usual amo


## 4. Tokenization: characters as tokens

The simplest possible tokenizer: every unique character in the text becomes a token. `vocab_size` is just the number of distinct characters.

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

 !.0123456789?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz~‘•■□
71


Now build two lookup tables — `stoi` (string to integer) and `itos` (integer to string) — to convert between characters and integer ids.

In [6]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("Hello there!"))
print(decode(encode("Hello there!")))

[21, 44, 51, 51, 54, 0, 59, 47, 44, 57, 44, 1]
Hello there!


## 5. Train / validation split

Encode the entire dataset into a tensor of integers, then hold out the last 10% as a validation set.

In [7]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(data.shape, data.dtype)
print(data[:100])

torch.Size([5991293]) torch.int64
tensor([33, 21, 18,  0, 15, 28, 38,  0, 36, 21, 28,  0, 25, 22, 35, 18, 17,  0,
        26, 57,  0, 40, 53, 43,  0, 26, 57, 58,  0, 17, 60, 57, 58, 51, 44, 64,
         0, 54, 45,  0, 53, 60, 52, 41, 44, 57,  0, 45, 54, 60, 57,  0, 29, 57,
        48, 61, 44, 59,  0, 17, 57, 48, 61, 44,  0, 62, 44, 57, 44,  0, 55, 57,
        54, 60, 43,  0, 59, 54,  0, 58, 40, 64,  0, 59, 47, 40, 59,  0, 59, 47,
        44, 64,  0, 62, 44, 57, 44,  0, 55, 44])


## 6. Batching

We never feed the whole dataset to the model at once. Instead we sample random chunks of length `block_size`. For each chunk `x`, the target `y` is the same chunk shifted one character to the right — i.e. "predict the next character at every position".

**In class:** fill in the `TODO`s below.

In [8]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data

    # TODO: sample `batch_size` random starting indices into `data`
    # (careful not to run past the end once you slice out block_size characters)
    # ix = ...

    # TODO: stack the block_size-length chunk starting at each index in ix -> x
    # x = ...

    # TODO: stack the same chunks shifted one character to the right -> y (the targets)
    # y = ...

    x, y = x.to(device), y.to(device)
    return x, y

In [9]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([32, 8])
tensor([[54, 60, 57,  0, 52, 54, 59, 47],
        [ 0, 52, 40, 64,  0, 58, 59, 48],
        [44, 58, 58, 54, 57,  0, 15, 51],
        [58, 40, 48, 43,  0, 19, 44, 44],
        [58, 59, 60, 43, 44, 53, 59, 58],
        [57, 40, 48, 59,  0, 58, 62, 60],
        [ 0, 40, 42, 42, 54, 52, 55, 40],
        [ 0, 40, 58,  0, 47, 44,  0, 43],
        [40, 50, 48, 53, 46,  0, 54, 60],
        [48, 58,  0, 48, 53,  0, 45, 40],
        [51, 40, 55,  0, 55, 60, 57, 57],
        [59,  0, 47, 48, 58,  0, 55, 54],
        [57, 44,  0, 47, 40, 43,  0, 59],
        [21, 40, 57, 57, 64,  0,  2, 27],
        [41, 51, 48, 53, 50, 48, 53, 46],
        [ 0, 41, 54, 64,  0, 47, 44, 58],
        [47, 54,  0, 62, 40, 58,  0, 58],
        [44,  0, 58, 52, 48, 57, 50, 44],
        [ 0, 59, 47, 44, 64,  0, 55, 40],
        [40, 57, 57, 64,  0, 51, 40, 60],
        [40, 51, 51, 64,  0, 21, 40, 46],
        [59,  0, 52, 44,  0, 52, 54, 58],
        [47, 40, 61, 44,  0, 43, 57, 40],
      

## 7. Estimating loss

Loss on a single batch is noisy, so we average over `eval_iters` batches to get a more stable estimate of train/val loss.

In [10]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## 8. The Bigram Language Model

The entire model is a single `nn.Embedding` table of shape `(vocab_size, vocab_size)`. Given a token, it directly looks up the logits for the next token — no attention, no context beyond the current character. This *is* a bigram model.

- `forward`: looks up logits for each token in the input, and (if targets are given) computes cross-entropy loss
- `generate`: autoregressively samples new tokens from the model's predicted distribution

**In class:** fill in the `TODO`s below.

In [11]:
# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # TODO: each token should directly read off the logits for the next token
        # from a lookup table -> use a single nn.Embedding of shape (vocab_size, vocab_size)
        self.token_embedding_table = ...

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        # TODO: look up the logits for every token in idx -> shape (B,T,C)
        logits = ...

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # TODO: reshape logits to (B*T, C) and targets to (B*T,),
            # then compute the cross-entropy loss between them
            logits = ...
            targets = ...
            loss = ...

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # TODO: get the predictions for the current context
            logits, loss = ...
            # TODO: focus only on the last time step -> becomes (B, C)
            logits = ...
            # TODO: apply softmax to turn logits into probabilities
            probs = ...
            # TODO: sample the next token index from the probability distribution
            idx_next = ...
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

## 9. Sanity check: untrained generation

Before training, the lookup table is random, so the model should generate gibberish. This gives us a baseline to compare against after training.

In [12]:
model = BigramLanguageModel(vocab_size)
model = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))

0.005041 M parameters
 6QXP1f6•3C3UypVLj6mIchDPS?~fl!Tu■4rJ!~w~40yrNpor8AZPJ!YevQvHD1ngNNJMirHMDYbD□Lo~59p.ZQ.JSG3dZqY.Sezfs‘H1pGRk6•gmvmyniwgm□sS~PiW6l6j5uk6j■WvmKCrdi~ m2sPPriuui4 ?•SDA0l6QvHMu□!d6V FHpwmGbbb2!Qzbbqawb?.j


## 10. Training the model

We use AdamW to update the embedding table so it learns the bigram statistics of the dataset.

In [13]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.7187, val loss 4.7118
step 500: train loss 4.1278, val loss 4.1140
step 1000: train loss 3.6476, val loss 3.6464
step 1500: train loss 3.2894, val loss 3.2740
step 2000: train loss 3.0306, val loss 3.0096
step 2500: train loss 2.8387, val loss 2.8205
step 3000: train loss 2.7053, val loss 2.6874
step 3500: train loss 2.6099, val loss 2.5982
step 4000: train loss 2.5432, val loss 2.5351
step 4500: train loss 2.5140, val loss 2.4879
step 5000: train loss 2.4805, val loss 2.4603
step 5500: train loss 2.4559, val loss 2.4408
step 6000: train loss 2.4283, val loss 2.4192
step 6500: train loss 2.4252, val loss 2.4022
step 7000: train loss 2.4283, val loss 2.4058
step 7500: train loss 2.4047, val loss 2.4021
step 8000: train loss 2.3976, val loss 2.3861
step 8500: train loss 2.3974, val loss 2.3932
step 9000: train loss 2.3960, val loss 2.3745
step 9500: train loss 2.3974, val loss 2.3747
step 9999: train loss 2.3839, val loss 2.3698


## 11. Generate text from the trained model

The model has now learned which character tends to follow which — let's see what it produces.

In [14]:
print('''\n##########################################
# Let's generate some Harry Potter text! #
##########################################''')
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=1000)[0].tolist()))


##########################################
# Let's generate some Harry Potter text! #
##########################################
 amebeth win elly ay y t watons it tart e .Z3Jzen thooneatrr t tle acelyit Ing ingelis fiseryowiffoonofeorg hithe n medksemWe ismyougrredf t e w as ve m ary whenthalle lckemars aneatetct ong scol .Wed He oung hearwenlyokes wh be thasarid longarwoftaimy wopand blthindefe so .Edong as im kild .SThin !46GAppppung ncl !sagheshede afof as t trorith arrintheeywanorstilwe he s jus Mane hot heredeven treveng Cedy hendinile fpory twhestlone hem t mus ped Hoverrt .Han f momal athed g thewe HAld alopuseday arerres atherid hed bermurd avete .Par sf t pstd uley pe .K y ss !Puscte jan ngheleatofthedousen .Whie te Q~ Weshein ommenthare wersly a Omyesushambort heaifAV7YofiQEFr t ne .Lot m.LGout sie h pe saHelyowinel hale ce o .Has pplye ad isag as ghe spt me .Nangsoiond s aw t in Pong d tle s istannealarothawing Ne athang Iforre utovearyod evisplid .Hask y in ascetapucre lo